# Baseline stream recommender

This notebook consumes `data/live_stream_features.jsonl`, normalizes the basic stream features, applies an explicit linear weight vector, and produces the top 50 live streams.

There are not yet interaction labels such as clicks or watch duration, so this is a coefficient-defined linear scoring baseline rather than a statistically fitted regression. The resulting `y_score` is the target to replace with a learned target when labels are available.

## Load exported data

Run `00_get_data.ipynb` first so that `data/live_stream_features.jsonl` exists.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

notebook_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
data_path = notebook_dir / 'data' / 'live_stream_features.jsonl'
if not data_path.exists():
    raise FileNotFoundError(f'Run 00_get_data.ipynb first: {data_path}')

streams_df = pd.read_json(data_path, lines=True)
print(f'Loaded {len(streams_df)} live streams from {data_path}')
streams_df.head(3)

Loaded 5000 live streams from /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data/live_stream_features.jsonl


,id,user_id,user_login,user_name,game_id,game_name,title,language,tags,is_mature,viewer_count,started_at,stream_age_hours,viewer_log1p,tag_count,title_length,is_followed_channel,matches_recent_vod_category
0,318188581220,92038375,caedrel,Caedrel,21779,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,en,"[English, LPL, LCK, LEC, Worlds, Esports, Drop...",False,209248,2026-10-31 13:00:36+00:00,4.510291,12.251280,7,35,False,False
1,318190806372,27115917,kamet0,Kamet0,21779,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,fr,"[Français, LEC, leagueoflegends, esport, RLCS,...",False,144733,2026-10-31 16:20:11+00:00,1.183903,11.882653,10,108,False,False
2,318190614756,83232866,ibai,ibai,21779,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,es,"[Español, IRL, Chatting, KOI, DropsActivados]",False,93931,2026-10-31 16:03:29+00:00,1.462236,11.450326,5,112,False,False


## Define and normalize the baseline features

All features are normalized to `[0, 1]` before weights are applied. Viewer count uses the API export's `viewer_log1p` feature so very large channels do not dominate linearly. Stream age is inverted because newer streams are preferred.

In [2]:
feature_columns = [
    'viewer_log1p',
    'tag_count',
    'title_length',
    'stream_age_hours',
    'is_followed_channel',
    'matches_recent_vod_category',
]

missing_columns = [column for column in feature_columns if column not in streams_df.columns]
if missing_columns:
    raise ValueError(f'Export is missing baseline features: {missing_columns}')

numeric_features = streams_df[feature_columns].copy()
for column in feature_columns:
    numeric_features[column] = pd.to_numeric(numeric_features[column], errors='coerce').fillna(0)

scaler = MinMaxScaler()
normalized = pd.DataFrame(
    scaler.fit_transform(numeric_features),
    columns=feature_columns,
    index=streams_df.index,
)
normalized['stream_age_hours'] = 1 - normalized['stream_age_hours']
normalized = normalized.clip(0, 1)
normalized.add_suffix('_normalized').head(3)

,viewer_log1p_normalized,tag_count_normalized,title_length_normalized,stream_age_hours_normalized,is_followed_channel_normalized,matches_recent_vod_category_normalized
0,1.000000,0.7,0.250000,0.906330,0.0,0.0
1,0.954901,1.0,0.771429,0.975837,0.0,0.0
2,0.902009,0.5,0.800000,0.970021,0.0,0.0


## Apply the linear baseline

Adjust these coefficients as the baseline evolves. Positive coefficients increase recommendation score; the coefficients sum to `1.0`, keeping `y_score` interpretable on `[0, 1]`.

In [3]:
baseline_weights = {
    'viewer_log1p': 0.50,
    'tag_count': 0.05,
    'title_length': 0.05,
    'stream_age_hours': 0.05,
    'is_followed_channel': 0.20,
    'matches_recent_vod_category': 0.15,
}

if not np.isclose(sum(baseline_weights.values()), 1.0):
    raise ValueError('Baseline weights must sum to 1.0')

weights = pd.Series(baseline_weights).reindex(feature_columns)
streams_scored_df = streams_df.copy()
streams_scored_df['y_score'] = normalized.dot(weights)
streams_scored_df = streams_scored_df.sort_values(
    ['y_score', 'viewer_count'], ascending=[False, False]
)

top_50_streams_df = streams_scored_df.head(50).reset_index(drop=True)
top_50_streams_df.insert(0, 'rank', top_50_streams_df.index + 1)
top_50_streams_df[['rank', 'user_name', 'game_name', 'title', 'viewer_count', 'y_score']].head(50)

,rank,user_name,game_name,title,viewer_count,y_score
0,1,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,144733,0.614814
1,2,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,209248,0.592816
2,3,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,93931,0.564505
3,4,otplol_,League of Legends,WORLDS - KC vs MKOI - POUR UNE PLACE EN QUART ...,48617,0.494661
4,5,Riot Games,League of Legends,KC vs MKOI | WORLDS 2026 | Swiss Stage Day 8,42921,0.482247
5,6,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,31078,0.482086
6,7,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],29750,0.462975
7,8,ow_esports,Overwatch,[DROPS] OWCS 2026 | Stage 3 Playoffs Day 2,21991,0.461264
8,9,morphe_ya,IRL,Забег день 4. ХЭЛЛОУИНСКИ МОРДЫ @dangerlyoha @...,23381,0.458722
9,10,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",27728,0.457499


In [4]:
output_path = notebook_dir / 'data' / 'baseline_top_50.jsonl'
top_50_streams_df.to_json(output_path, orient='records', lines=True, date_format='iso')
print(f'Wrote {len(top_50_streams_df)} ranked streams to {output_path}')
print('Weights:', baseline_weights)

Wrote 50 ranked streams to /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data/baseline_top_50.jsonl
Weights: {'viewer_log1p': 0.5, 'tag_count': 0.05, 'title_length': 0.05, 'stream_age_hours': 0.05, 'is_followed_channel': 0.2, 'matches_recent_vod_category': 0.15}
